# ST456 Deep Learning — PCam Histopathology Classification


## Section 0 — Setup cunks run this section first

In [ ]:
# Mount Drive and clone repo
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!git clone https://github.com/Friedrich-233/ST456_GroupProject.git /content/ST456_GroupProject
!pip install timm -q
import sys
sys.path.insert(0, '/content/ST456_GroupProject/code')

In [ ]:
from pathlib import Path

DRIVE_DATA_DIR  = Path('/content/drive/MyDrive/ST456 Group project/pcamv1')
DATA_DIR         = Path('/content/pcam_data')
CHECKPOINT_DIR   = Path('/content/ST456_GroupProject/checkpoints')
RESULTS_DIR      = Path('/content/ST456_GroupProject/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Checkpoints: {CHECKPOINT_DIR}")
print(f"Results:     {RESULTS_DIR}")

In [ ]:
from data import load_all_data, build_downstream_loaders, LABEL_FRACTIONS

print("Loading PCam data...")
data = load_all_data(
    drive_data_dir=DRIVE_DATA_DIR,
    data_dir=DATA_DIR,
    pretrain_fraction=0.15,
    downstream_pool_fraction=0.15,
    use_subset=True,
)

train_loaders, val_loader, test_loader = build_downstream_loaders(data)
print("\nLabel fractions:", list(LABEL_FRACTIONS.keys()))
print("Train loaders:", {k: len(v.dataset) for k, v in train_loaders.items()})

## Section 1 — Pre-training SSL Encoders

In [ ]:
# ═══════════════════════════════════════════════════════
# SMOKE TEST — Section 1 (SSL pretraining sanity check)
# Runs SimCLR + MAE each for 2 epochs to verify:
#   - Data loading works
#   - New params (AdamW, cosine schedule, warmup) flow correctly
#   - Checkpoints save successfully
# Expected runtime: ~3-5 min on Colab T4
# ═══════════════════════════════════════════════════════
from training import train_simclr, train_mae
from pathlib import Path

SMOKE_CKPT_DIR = Path('/tmp/smoke_test')
SMOKE_CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(">>> [1/2] SimCLR smoke test (2 epochs)")
_m, h, _ = train_simclr(
    data=data,
    checkpoint_dir=SMOKE_CKPT_DIR,
    epochs=2,
    warmup_epochs=1,
    tailored=True,
    checkpoint_name='smoke_simclr.pth',
)
assert len(h) == 2 and 'lr' in h[0], "SimCLR history should have 2 epochs with lr"
assert h[1]['loss'] < h[0]['loss'] + 0.5, "SimCLR loss should not explode"
print(f"  ✓ SimCLR OK — loss {h[0]['loss']:.3f} → {h[1]['loss']:.3f}, lr logged")

print("\n>>> [2/2] MAE smoke test (2 epochs)")
_m, h, _ = train_mae(
    data=data,
    checkpoint_dir=SMOKE_CKPT_DIR,
    epochs=2,
    warmup_epochs=1,
    checkpoint_name='smoke_mae.pth',
)
assert len(h) == 2 and 'lr' in h[0], "MAE history should have 2 epochs with lr"
assert h[1]['loss'] < h[0]['loss'] + 0.1, "MAE loss should not explode"
print(f"  ✓ MAE OK — loss {h[0]['loss']:.3f} → {h[1]['loss']:.3f}, lr logged")

print("\n" + "="*50)
print("✓ SECTION 1 SMOKE TEST PASSED — ready to run full pre-training")
print("="*50)

In [ ]:
from training import train_simclr, SEED

# Improve SimCLR tailored — 20 epochs, AdamW + cosine schedule + warmup
simclr_model, simclr_history, simclr_ckpt = train_simclr(
    data=data,
    checkpoint_dir=CHECKPOINT_DIR,
    epochs=20,                          
    batch_size=128,
    learning_rate=3e-4,
    warmup_epochs=2,                    
    weight_decay=0.05,                  
    tailored=True,
    checkpoint_name='simclr_encoder_tailored.pth',
)
print(f"\nSimCLR (tailored) checkpoint saved → {simclr_ckpt}")

In [ ]:
# Plot SimCLR training loss
import pandas as pd, matplotlib.pyplot as plt

df = pd.DataFrame(simclr_history)
plt.figure(figsize=(8, 4))
plt.plot(df['epoch'], df['loss'], 'b-o')
plt.xlabel('Epoch'); plt.ylabel('NT-Xent Loss')
plt.title('SimCLR Training Curve')
plt.grid(True, alpha=0.3); plt.tight_layout()
plt.show()

In [ ]:
# SimCLR GENERIC augmentation (ablation vs tailored)
simclr_model_generic, simclr_history_generic, simclr_ckpt_generic = train_simclr(
    data=data,
    checkpoint_dir=CHECKPOINT_DIR,
    epochs=20,
    batch_size=128,
    learning_rate=3e-4,
    warmup_epochs=2,
    weight_decay=0.05,
    tailored=False,                     # tailored to ablation
    checkpoint_name='simclr_encoder_generic.pth',
)
print(f"\nSimCLR (generic) checkpoint saved → {simclr_ckpt_generic}")

In [ ]:
from training import train_mae

# MAE Improve — 20 epochs, lr=1.5e-4 (MAE paper's linear scaling), cosine + warmup
mae_model, mae_history, mae_ckpt = train_mae(
    data=data,
    checkpoint_dir=CHECKPOINT_DIR,
    epochs=20,                         
    batch_size=128,
    learning_rate=1.5e-4,               
    warmup_epochs=2,                    
    weight_decay=0.05,                  
    checkpoint_name='mae_encoder.pth',
)
print(f"\nMAE checkpoint saved → {mae_ckpt}")

In [ ]:
# Plot MAE training loss
df = pd.DataFrame(mae_history)
plt.figure(figsize=(8, 4))
plt.plot(df['epoch'], df['loss'], 'r-o')
plt.xlabel('Epoch'); plt.ylabel('Reconstruction Loss')
plt.title('MAE (Base) Training Curve')
plt.grid(True, alpha=0.3); plt.tight_layout()
plt.show()

In [ ]:
import os

print("=" * 55)
print("  PRE-TRAINING COMPLETE — Checkpoints")
print("=" * 55)

for name, path in [
    ('SimCLR (tailored)',   CHECKPOINT_DIR / 'simclr_encoder_tailored.pth'),
    ('SimCLR (generic)',    CHECKPOINT_DIR / 'simclr_encoder_generic.pth'),   # Generic
    ('MAE (base)',          CHECKPOINT_DIR / 'mae_encoder.pth'),
]:
    status = '✓' if path.exists() else '✗ MISSING'
    if path.exists():
        size_mb = os.path.getsize(path) / 1e6
        print(f"  {status}  {name:<22}  {size_mb:.1f} MB  {path}")
    else:
        print(f"  {status}  {name:<22}  (not found)")

## Section 2 — Downstream Experiment Grid


In [ ]:
# ═══════════════════════════════════════════════════════
# SMOKE TEST — Section 2 (downstream + PEFT sanity check)
# Runs one tiny experiment per (method, strategy) combination.
# Uses 1% labels + 2 epochs to stay fast.
# Expected runtime: ~3-5 min on Colab T4
# ═══════════════════════════════════════════════════════
from training import run_single_experiment, TrainConfig
from models import count_trainable_params, LoRAResNetClassifier, VPTMAEClassifier

smoke_config = TrainConfig(epochs=2, patience=5)
smoke_cases = [
    ('supervised_from_scratch', 'full'),
    ('simclr', 'frozen'),
    ('mae', 'frozen'),
    ('simclr', 'peft'),              # LoRA
    ('mae', 'peft'),                 # VPT
]

print(f">>> Running {len(smoke_cases)} smoke cases (1% labels, 2 epochs each)")
for method, strategy in smoke_cases:
    r = run_single_experiment(
        method=method, strategy=strategy, label_name='1%',
        train_loader=train_loaders['1%'],
        val_loader=val_loader, test_loader=test_loader,
        simclr_checkpoint=CHECKPOINT_DIR / 'simclr_encoder_tailored.pth',
        mae_checkpoint=CHECKPOINT_DIR / 'mae_encoder.pth',
        seed=42, config=smoke_config,
    )
    assert 0.4 < r['test_auc'] < 1.0, f"{method}/{strategy} AUC abnormal: {r['test_auc']}"
    print(f"  ✓ {method:<24} {strategy:<7} → test_auc={r['test_auc']:.3f}")

print("\n" + "="*50)
print("✓ SECTION 2 SMOKE TEST PASSED — ready to run full grid (incl. PEFT)")
print("="*50)

In [ ]:
# ── Section 2 imports ─────────────────────────────────────────────────────────
# Run Section 0 first (cells 1-3) so that CHECKPOINT_DIR, RESULTS_DIR,
# data, train_loaders, val_loader, test_loader are all in scope.
import os
from training import (
    run_single_experiment,
    run_experiment_grid,
    TrainConfig,
    DEFAULT_EXPERIMENT_CONFIG,
    MAIN_METHODS,
    FINETUNE_STRATEGIES,
)

# Verify checkpoints produced by Section 1 are present
_ckpts = {
    'simclr_encoder_tailored.pth': CHECKPOINT_DIR / 'simclr_encoder_tailored.pth',
    'simclr_encoder_generic.pth':  CHECKPOINT_DIR / 'simclr_encoder_generic.pth',  # ← 加
    'mae_encoder.pth':             CHECKPOINT_DIR / 'mae_encoder.pth',
}
for _name, _path in _ckpts.items():
    if not _path.exists():
        raise FileNotFoundError(
            f"Checkpoint missing: {_path}\n"
            "Run Section 1 first, or copy the checkpoints into CHECKPOINT_DIR."
        )
    print(f"  ✓  {_name}")
print("All checkpoints found — ready to run experiments.")

In [ ]:
# Single test run
test_result = run_single_experiment(
    method='mae',                     
    strategy='frozen',
    label_name='1%',
    train_loader=train_loaders['1%'],
    val_loader=val_loader,
    test_loader=test_loader,
    mae_checkpoint=CHECKPOINT_DIR / 'mae_encoder.pth',   
    seed=42,
)
print(f"\nTest run — MAE, frozen, 1%:") 
print(f"  test_auc = {test_result['test_auc']:.4f}")
print(f"  test_acc = {test_result['test_accuracy']:.4f}")
print(f"  test_f1  = {test_result['test_f1']:.4f}")

In [ ]:
# Grid A — Seed 42
results_42 = run_experiment_grid(
    methods=MAIN_METHODS,
    strategies=FINETUNE_STRATEGIES,
    label_names=list(LABEL_FRACTIONS.keys()),
    seeds=[42],
    train_loaders=train_loaders,
    val_loader=val_loader,
    test_loader=test_loader,
    simclr_checkpoint=CHECKPOINT_DIR / 'simclr_encoder_tailored.pth',
    mae_checkpoint=CHECKPOINT_DIR / 'mae_encoder.pth',
    config=DEFAULT_EXPERIMENT_CONFIG,
)
results_42.to_csv(RESULTS_DIR / 'results_seed42.csv', index=False)
print(f"Seed 42 done — {len(results_42)} rows → {RESULTS_DIR / 'results_seed42.csv'}")
display(results_42)


In [ ]:
# Grid B — Seed 52
results_52 = run_experiment_grid(
    methods=MAIN_METHODS,
    strategies=FINETUNE_STRATEGIES,
    label_names=list(LABEL_FRACTIONS.keys()),
    seeds=[52],
    train_loaders=train_loaders,
    val_loader=val_loader,
    test_loader=test_loader,
    simclr_checkpoint=CHECKPOINT_DIR / 'simclr_encoder_tailored.pth',
    mae_checkpoint=CHECKPOINT_DIR / 'mae_encoder.pth',
    config=DEFAULT_EXPERIMENT_CONFIG,
)
results_52.to_csv(RESULTS_DIR / 'results_seed52.csv', index=False)
print(f"Seed 52 done — {len(results_52)} rows → {RESULTS_DIR / 'results_seed52.csv'}")
display(results_52)


In [ ]:
# Grid C — Seed 62
results_62 = run_experiment_grid(
    methods=MAIN_METHODS,
    strategies=FINETUNE_STRATEGIES,
    label_names=list(LABEL_FRACTIONS.keys()),
    seeds=[62],
    train_loaders=train_loaders,
    val_loader=val_loader,
    test_loader=test_loader,
    simclr_checkpoint=CHECKPOINT_DIR / 'simclr_encoder_tailored.pth',
    mae_checkpoint=CHECKPOINT_DIR / 'mae_encoder.pth',
    config=DEFAULT_EXPERIMENT_CONFIG,
)
results_62.to_csv(RESULTS_DIR / 'results_seed62.csv', index=False)
print(f"Seed 62 done — {len(results_62)} rows → {RESULTS_DIR / 'results_seed62.csv'}")
display(results_62)


In [ ]:
# Merge all seeds and save final results
import pandas as pd

results_df = pd.concat([results_42, results_52, results_62], ignore_index=True)
results_df.to_csv(RESULTS_DIR / 'results_main.csv', index=False)
print(f"Combined: {len(results_df)} rows (3 seeds × 3 methods × 3 strategies × 3 label fractions)")
print(f"Saved → {RESULTS_DIR / 'results_main.csv'}")
display(results_df)

In [ ]:
# Run this only if you trained a 'generic' SimCLR checkpoint as well.
# Point simclr_generic_checkpoint to wherever you saved it.

simclr_generic_checkpoint = CHECKPOINT_DIR / 'simclr_encoder_generic.pth'

if simclr_generic_checkpoint.exists():
    results_simclr_generic = run_experiment_grid(
        methods=['simclr'],
        strategies=FINETUNE_STRATEGIES,
        label_names=list(LABEL_FRACTIONS.keys()),
        seeds=[42, 52, 62],
        train_loaders=train_loaders,
        val_loader=val_loader,
        test_loader=test_loader,
        simclr_checkpoint=simclr_generic_checkpoint,
        mae_checkpoint=None,
        config=DEFAULT_EXPERIMENT_CONFIG,
    )
    results_simclr_generic.to_csv(RESULTS_DIR / 'results_simclr_generic.csv', index=False)
    print(f"SimCLR generic ablation saved → {RESULTS_DIR / 'results_simclr_generic.csv'}")
else:
    print('Generic SimCLR checkpoint not found — skipping ablation.')

In [ ]:
from evaluation import summarise_results, build_report_table

summary = summarise_results(results_df)
report_table = build_report_table(summary)
display(report_table)

## Section 3 — Results Analysis


In [ ]:
# ── Section 3 imports & data loading ─────────────────────────────────────────
# Run Section 0 first (cells 1-3) so that RESULTS_DIR and CHECKPOINT_DIR
# are in scope.  If Section 2 already ran, results_df is reused from memory;
# otherwise it is loaded from the saved CSV.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

FIGURE_DIR = Path('/content/figures')
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Load results — prefer in-memory DataFrame from Section 2, else from CSV
try:
    results_df          # already defined by Section 2
    print(f"Using in-memory results_df ({len(results_df)} rows).")
except NameError:
    _csv = RESULTS_DIR / 'results_main.csv'
    if not _csv.exists():
        raise FileNotFoundError(
            f"Results CSV not found: {_csv}\n"
            "Run Section 2 first to generate experiment results."
        )
    results_df = pd.read_csv(_csv)
    print(f"Loaded {len(results_df)} rows from {_csv}")

from evaluation import summarise_results, build_report_table
summary      = summarise_results(results_df)
report_table = build_report_table(summary)
print("Summary table ready.")

In [ ]:
from evaluation import summarise_results, build_report_table

summary      = summarise_results(results_df)
report_table = build_report_table(summary)

# Save
report_table.to_csv(RESULTS_DIR / 'report_table.csv', index=False)
print("Report table saved → results/report_table.csv")
display(report_table)

In [ ]:
# Pivot for heatmap: rows = method, cols = (strategy × label_fraction)
heatmap_data = (
    summary[["method", "strategy", "label_fraction", "test_auc"]]
    .assign(label_fraction=summary[('test_auc', 'label_fraction')])
    # Pivot manually
)

# Use the mean column
auc_mean = summary[('test_auc', 'mean')].values.reshape(3, 9)  
methods = summary[('method', '')].unique()[:3]         
labels  = [f"{s} / {l}" for s, l in zip(
    summary[('strategy', '')],
    summary[('test_auc', 'label_fraction')]
)]

import numpy as np
methods_arr = summary.groupby('method').first().index.tolist()[:3]
strategies  = ['frozen', 'partial', 'full']
label_list  = ['1%', '5%', '10%']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, label in zip(axes, label_list):
    mask = summary[('test_auc', 'label_fraction')] == label
    sub  = summary[mask].set_index('method').loc[methods_arr]
    matrix = sub[[('test_auc', 'mean')]].values.reshape(3, 3).T  # strategy × method
    sns.heatmap(
        matrix,
        annot=True, fmt='.3f',
        xticklabels=strategies,
        yticklabels=methods_arr,
        ax=ax,
        vmin=0.65, vmax=0.95,
        cmap='YlOrRd'
    )
    ax.set_title(f'Test AUC — {label} labelled')
    ax.set_xlabel('Fine-tune strategy')
    ax.set_ylabel('Method')

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'auc_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot val AUC progression across label fractions for each method
# We use the first seed's history from run_single_experiment
# If you re-ran with multiple seeds, extract history from the saved dicts.

fig, axes = plt.subplots(1, 3, figsize=(15, 5))          # ← 2×2 to 1×3

for ax, method in zip(axes, ['supervised_from_scratch', 'simclr', 'mae']): 
    sub = results_df[results_df['method'] == method]
    for label in ['1%', '5%', '10%']:
        rows = sub[sub['label_fraction'] == label]
        means = rows.groupby('seed')['best_val_auc'].mean()
        # Show spread across seeds via a bar
        aucs = rows.groupby('seed')['test_auc'].mean()
        ax.bar(label, aucs.mean(), yerr=aucs.std(), capsize=5,
               label=label, alpha=0.8)
    ax.set_title(method)
    ax.set_xlabel('Label fraction')
    ax.set_ylabel('Test AUC')
    ax.set_ylim(0.5, 1.0)
    ax.legend(title='Label frac')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Test AUC by Method × Label Fraction', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'auc_by_method_label.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Load the full y_true / y_prob arrays from the best seed run.
# The raw arrays are stored in the saved result dicts.
# Here we re-compute them from the final model of the best config.

from data import load_all_data, build_downstream_loaders, LABEL_FRACTIONS
from training import build_simclr_classifier, build_supervised_scratch_classifier
from training import build_mae_classifier
from training import train_classifier, DEFAULT_EXPERIMENT_CONFIG, set_seed
from evaluation import evaluate_model

DRIVE_DATA_DIR = Path('/content/drive/MyDrive/ST456 Group project/pcamv1')
DATA_DIR        = Path('/content/pcam_data')

print('Loading data for best-model evaluation...')
data = load_all_data(DRIVE_DATA_DIR, DATA_DIR,
                     pretrain_fraction=0.15,
                     downstream_pool_fraction=0.15,
                     use_subset=True)
train_loaders, val_loader, test_loader = build_downstream_loaders(data)

best_configs = [
    ('supervised_from_scratch', 'full',   None),                                           # ← 之前的 ckpt path 其实是 bug
    ('simclr',                  'frozen', CHECKPOINT_DIR / 'simclr_encoder_tailored.pth'),
    ('mae',                     'frozen', CHECKPOINT_DIR / 'mae_encoder.pth'),
]
set_seed(42)
roc_results = {}

for method, strategy, ckpt_path in best_configs:
    print(f'\nEvaluating {method} / {strategy}...')
    if method == 'supervised_from_scratch':
        model = build_supervised_scratch_classifier()
    elif method == 'simclr':
        model = build_simclr_classifier(ckpt_path)
    elif method == 'mae':
        model = build_mae_classifier(ckpt_path)
    else:
        raise ValueError(f"Unknown method: {method}")

    model, _, _, _ = train_classifier(
        model, train_loaders['10%'], val_loader, strategy=strategy,
        config=DEFAULT_EXPERIMENT_CONFIG,
    )
    metrics = evaluate_model(model, test_loader)
    roc_results[method] = {
        'y_true': metrics['y_true'],
        'y_prob': metrics['y_prob'],
        'test_auc': metrics['auc'],
    }
    print(f'  test_auc = {metrics["auc"]:.4f}')

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, auc

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['gray', 'blue', 'red']

for (method, res), color in zip(roc_results.items(), colors):
    fpr, tpr, _ = roc_curve(res['y_true'], res['y_prob'])
    precision, recall, _ = precision_recall_curve(res['y_true'], res['y_prob'])
    axes[0].plot(fpr, tpr, color=color, lw=2,
                  label=f"{method} (AUC={res['test_auc']:.3f})")
    axes[1].plot(recall, precision, color=color, lw=2,
                  label=f"{method} (AUC={res['test_auc']:.3f})")

axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves (10% labelled, full fine-tune)')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].grid(alpha=0.3)

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves (10% labelled)')
axes[1].legend(loc='upper right', fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from evaluation import plot_tsne

set_seed(42)

for method, strategy, ckpt_path in best_configs:
    if method == 'supervised_from_scratch':
        model = build_supervised_scratch_classifier()
    elif method == 'simclr':
        model = build_simclr_classifier(ckpt_path)
    elif method == 'mae':
        model = build_mae_classifier(ckpt_path)
    else:
        raise ValueError(f"Unknown method: {method}")

    model, _, _, _ = train_classifier(
        model, train_loaders['10%'], val_loader, strategy=strategy,
        config=DEFAULT_EXPERIMENT_CONFIG,
    )
    embeddings, labels = extract_embeddings(model, test_loader, method)
    plot_tsne(embeddings, labels, title=f'{method} t-SNE')
    plt.savefig(FIGURE_DIR / f'tsne_{method}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f't-SNE for {method} saved.')

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from evaluation import GradCAM
from data import make_classification_transform

set_seed(42)

# Reload the two models we want to compare
model_sup = build_supervised_scratch_classifier()
model_sup, _, _, _ = train_classifier(
    model_sup, train_loaders['10%'], val_loader,
    strategy='full', config=DEFAULT_EXPERIMENT_CONFIG,
)

model_mae = build_mae_classifier(
    CHECKPOINT_DIR / 'mae_encoder.pth'
)
model_mae, _, _, _ = train_classifier(
    model_mae, train_loaders['10%'], val_loader,
    strategy='partial', config=DEFAULT_EXPERIMENT_CONFIG,
)

transform = make_classification_transform(data.channel_mean, data.channel_std)

# Pick 4 test images
indices = [0, 10, 50, 100]
fig, axes = plt.subplots(4, 5, figsize=(18, 14))

for row, idx in enumerate(indices):
    img = data.x_test[idx]          # HWC float [0,1]
    label = int(data.y_test[idx])
    img_tensor = transform(img).unsqueeze(0).to('cuda' if torch.cuda.is_available() else 'cpu')

    # Row: original, supervised-GC, mae-GC, supervised-GC overlay, mae-GC overlay
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f'True label: {label}')
    axes[row, 0].axis('off')

    # Supervised Grad-CAM
    cam_sup = GradCAM(model_sup.cuda(), model_sup.encoder.layer4)
    heat_sup = cam_sup(img_tensor.cuda(), class_idx=label)

    axes[row, 1].imshow(img)
    axes[row, 1].imshow(heat_sup, cmap='jet', alpha=0.4)
    axes[row, 1].set_title('Supervised (full) GC')
    axes[row, 1].axis('off')

    # MAE Improved Grad-CAM — Grad-CAM works on conv layers; MAE encoder is ViT-based.
    # We attach to the last decoder conv / patch embed instead.
    cam_mae = GradCAM(model_mae.cuda(), model_mae.encoder.patch_embed)
    heat_mae = cam_mae(img_tensor.cuda(), class_idx=label)

    axes[row, 2].imshow(img)
    axes[row, 2].imshow(heat_mae, cmap='jet', alpha=0.4)
    axes[row, 2].set_title('MAE GC')
    axes[row, 2].axis('off')

plt.suptitle('Grad-CAM Comparison: Supervised vs MAE')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'gradcam_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import os

print('=' * 55)
print('  ANALYSIS COMPLETE — Saved outputs')
print('=' * 55)
print(f'\nResults:')
for f in sorted(os.listdir(RESULTS_DIR)):
    print(f'  {RESULTS_DIR}/{f}')

print(f'\nFigures:')
for f in sorted(os.listdir(FIGURE_DIR)):
    size = os.path.getsize(FIGURE_DIR / f) / 1e3
    print(f'  {FIGURE_DIR}/{f}  ({size:.0f} KB)')

print('\nAll done. These figures and tables can be used directly in the report.')